Cell 1 Import Library

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

Cell 2 Load and Preprocessing

In [26]:
from pandas import errors
df = pd.read_csv('../data/gempa_1990-2026.csv')
df = df[df['type'] == 'earthquake'].copy()

df['time'] = pd.to_datetime(df['time'])
df['bulan'] = df['time'].dt.month
df['jam'] = df['time'].dt.hour

fitur = ['latitude', 'longitude', 'depth', 'gap', 'dmin', 'nst', 'bulan', 'jam']

for col in ['gap', 'dmin', 'nst']:
    df[col] = pd.to_numeric(df[col], errors='coerce') #ini kalo ada yang kososng coerce mengisi NaN kalo gak salah
    df[col] = df[col].fillna(df[col].median()) #ini untuk pengisian mediannya

df_clean = df[fitur].dropna().reset_index(drop=True) #ini biar indexnya rapih dan data tidak loncat ke tempat yang salah. karna index tidak sekedar penamaan doang

print(f"Total data: {df_clean.shape[0]} baris")
df_clean.head()


Total data: 63413 baris


,latitude,longitude,depth,gap,dmin,nst,bulan,jam
0,-5.092,103.514,99.0,101.0,2.082,24.0,12,21
1,-9.697,124.116,78.5,101.0,2.082,24.0,12,3
2,-8.608,119.387,142.2,101.0,2.082,24.0,12,6
3,-5.112,102.689,60.7,101.0,2.082,24.0,12,4
4,5.211,126.316,72.5,101.0,2.082,24.0,12,22


Cell 3 Isolation Forest (Generate Label)

In [30]:
from scipy.sparse import random
X_raw = df_clean[fitur]

X_train_raw, X_test_raw = train_test_split(
    X_raw, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train_raw):,} data | Test: {len(X_test_raw):,} data | Total: {len(X_raw):,} data")


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.fit_transform(X_test_raw)

#test menggunakan 3 nilai contamination
hasil_if_train = {}
hasil_if_test = {}
for c in [0.01, 0.05, 0.10]:
    iso = IsolationForest(contamination=c, n_estimators=100, random_state=42)
    iso.fit(X_train_scaled)
    label_train = (iso.predict(X_train_scaled) == -1).astype(int)
    label_test = (iso.predict(X_test_scaled) == -1).astype(int)
    hasil_if_train[c] = label_train
    hasil_if_test[c] = label_test
    print(f"Contamination {c}: train {label_train.sum()} anomali, test {label_test.sum()} anomali")




Train: 50,730 data | Test: 12,683 data | Total: 63,413 data
Contamination 0.01: train 508 anomali, test 133 anomali
Contamination 0.05: train 2537 anomali, test 653 anomali
Contamination 0.1: train 5073 anomali, test 1314 anomali
